In [ ]:
import genanki
import random
import csv
from pathlib import Path
import os

# CSV to APKG Converter

This notebook converts CSV vocabulary files into Anki package (.apkg) files.

## Usage Instructions

1. **Add CSV files**: Put your CSV files in the `input/` folder
2. **Run cells**: Execute all cells above in order
3. **Get results**: Find your `.apkg` files in the `output/` folder

### CSV Format
Your CSV files should have two columns:
- Column 1: Word (front of flashcard)
- Column 2: Meaning (back of flashcard)

Example:
```
apple,red fruit
book,reading material
computer,electronic device
```s


In [ ]:
# Create Anki card model
def create_anki_model():
    """Create a basic Anki card model for vocabulary"""
    return genanki.Model(
        random.randrange(1 << 30, 1 << 31),  # Random model ID
        'Vocabulary Model',
        fields=[
            {'name': 'Word'},
            {'name': 'Meaning'},
        ],
        templates=[
            {
                'name': 'Card 1',
                'qfmt': '<div style="font-size: 24px; text-align: center;">{{Word}}</div>',
                'afmt': '{{FrontSide}}<hr id="answer"><div style="font-size: 18px; text-align: center;">{{Meaning}}</div>',
            },
        ],
        css="""
        .card {
            font-family: Arial, sans-serif;
            background-color: #f9f9f9;
            padding: 20px;
        }
        """
    )

model = create_anki_model()
print("✅ Anki model created")


In [ ]:
# Set up directories
input_dir = Path("input")
output_dir = Path("output")

# Create directories if they don't exist
input_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

print(f"📁 Input directory: {input_dir.absolute()}")
print(f"📁 Output directory: {output_dir.absolute()}")


In [ ]:
# Find all CSV files in input directory
csv_files = list(input_dir.glob("*.csv"))

if csv_files:
    print(f"Found {len(csv_files)} CSV file(s):")
    for csv_file in csv_files:
        print(f"  📄 {csv_file.name}")
else:
    print("⚠️ No CSV files found in input directory")
    print("Please add CSV files to the input folder")


In [ ]:
# Function to convert CSV to APKG
def csv_to_apkg(csv_file_path, output_dir, model):
    """Convert a single CSV file to APKG format"""
    print(f"\n🔄 Processing: {csv_file_path.name}")
    
    # Create deck name from filename
    deck_name = csv_file_path.stem
    deck_id = random.randrange(1 << 30, 1 << 31)
    
    # Create deck
    deck = genanki.Deck(deck_id, deck_name)
    
    # Read CSV and create notes
    try:
        with open(csv_file_path, 'r', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            note_count = 0
            
            for row_num, row in enumerate(reader, 1):
                if len(row) >= 2:  # Ensure we have at least 2 columns
                    word = row[0].strip()
                    meaning = row[1].strip()
                    
                    if word and meaning:  # Skip empty rows
                        note = genanki.Note(
                            model=model,
                            fields=[word, meaning]
                        )
                        deck.add_note(note)
                        note_count += 1
                        print(f"  ✓ Row {row_num}: {word} -> {meaning}")
                    else:
                        print(f"  ⚠️ Row {row_num}: Skipped empty row")
                else:
                    print(f"  ⚠️ Row {row_num}: Not enough columns")
            
            if note_count > 0:
                # Generate output filename
                output_file = output_dir / f"{deck_name}.apkg"
                
                # Create package and write to file
                package = genanki.Package(deck)
                package.write_to_file(str(output_file))
                
                print(f"✅ Created: {output_file} ({note_count} cards)")
                return True, note_count
            else:
                print(f"❌ No valid cards found in {csv_file_path}")
                return False, 0
                
    except Exception as e:
        print(f"❌ Error processing {csv_file_path}: {str(e)}")
        return False, 0

print("CSV to APKG conversion function ready")
